# Vraagdrijvers vinden met PROC GLMSELECT: Stapsgewijs, LASSO en gevalideerde voorwaartse selectie

## Samenvatting

Een retailanalyseteam gebruikt **PROC GLMSELECT** om te ontdekken welke marketing- en prijshendels de wekelijkse verkochte eenheden echt beïnvloeden, en om echte vraagdrijvers te scheiden van ruis. Stapsgewijze selectie beoordeeld op SBC, LASSO met 5-voudige kruisvalidatie en een op een holdout gevalideerde voorwaartse zoektocht herstellen alle drie **dezelfde set van echte drijvers** — eigen prijs, concurrentieprijs, advertentie-uitgaven, e-mailvolume, promoties, feestdagen, de regio Northeast en het kanaal Online — en elk van de vier ingebouwde afleiders (`temp_f`, `noise1`, `noise2`, `noise3`) wordt automatisch verwijderd.

De methoden komen ook nauw overeen wat de grootteordes betreft: elk schat een eigen-prijseffect rond **-28 eenheden per dollar** en een concurrentieprijseffect rond **+14**, precies de substitutietekens die in de data-genererende vergelijking zijn ingebouwd. Het enige verschil zit aan de rand — de kruisgevalideerde LASSO behoudt bovendien een klein, statistisch verwaarloosbaar `Region=Midwest`-contrast (schatting -8,3 met een standaardfout van 8,3) dat zowel stapsgewijze als voorwaartse selectie laten vallen. Een drijverlijst die drie verschillende selectiefilosofieën overleeft, is veel betrouwbaarder dan een die op één enkele regel is afgestemd.

## Gegevensbronnen

Alle gegevens in dit notebook zijn **synthetisch** en inline gegenereerd (geen externe bestanden, seed `20260531`). Ze bootsen één seizoen van winkel-week vraagpanels na voor een detailhandelaar in consumentengoederen.

| Dataset | Rijen | Granulariteit | Kernvariabelen |
|---------|------|-------|---------------|
| `demand` | 100 | Winkel-week | `units` (respons: wekelijks verkochte eenheden); `price`, `competitor` (eigen & concurrerende schapprijs); `adspend`, `email` (mediadruk); `promo`, `holiday` (gebeurtenisindicatoren); `region`, `channel` (CLASS-effecten); `temp_f`, `noise1`-`noise3` (afleider/irrelevante voorspellers) |

`units` wordt opgebouwd uit een bekende lineaire vergelijking, zodat de "juiste" set van drijvers verifieerbaar is; `temp_f` en de drie `noise`-kolommen dragen geen echt signaal en dienen om te testen of elke selectiemethode ze verwijdert.

# Vraagdrijvers vinden met PROC GLMSELECT

Een categoriemanager heeft tientallen kandidaat-verklarende variabelen voor de wekelijkse verkoop: eigen prijs, concurrentieprijs, advertentie-uitgaven, e-mailvolume, promoties, feestdagen, winkelregio, verkoopkanaal, zelfs het weer. Ze allemaal in één regressie stoppen nodigt uit tot overfitting en instabiele coëfficiënten. **PROC GLMSELECT** automatiseert de zoektocht naar een spaarzaam model, met ondersteuning voor stapsgewijze selectie, LASSO, elastic net en least-angle-selectie met ingebouwde kruisvalidatie en holdout-partitionering.

In dit notebook doen we het volgende:

1. Genereer een realistisch synthetisch winkel-week vraagpanel met een *bekende* set van echte drijvers (plus opzettelijke afleidervariabelen).
2. Voer **stapsgewijze selectie** uit, beoordeeld op SBC.
3. Voer **LASSO** uit met 5-voudige kruisvalidatie.
4. Voer **voorwaartse selectie gevalideerd op een 30% holdout** uit.

Een goede selectiemethode zou de echte drijvers moeten herstellen en de ruis moeten verwijderen — laten we kijken.

## 1. Genereer een synthetisch vraagpanel

De respons `units` wordt opgebouwd uit een expliciete lineaire vergelijking, zodat we de grondwaarheid kennen: prijs en concurrentieprijs, advertentie-uitgaven, e-mailvolume, de promo- en feestdagindicatoren, plus regio- en kanaaleffecten doen er allemaal toe. De variabelen `temp_f`, `noise1`, `noise2` en `noise3` zijn pure afleiders zonder echte relatie met de verkoop. Een `call streaminit`-seed maakt de gegevens reproduceerbaar.

In [1]:
/* ---------------------------------------------------------------
   Synthetisch winkel-week vraagpanel voor een detailhandelaar in consumentengoederen.
   'units' volgt een BEKENDE vergelijking; temp_f en noise1-3 zijn afleiders.
   --------------------------------------------------------------- */
GEGEVENS demand;
    CALL streaminit(20260531);
    LENGTE region $9 channel $8 promo $3;
    DOE store_week = 1 TOT 100;
        /* Regiomix */
        u1 = rand('uniform');
        ALS u1 < 0.34 DAN region = 'Northeast';
        ANDERS ALS u1 < 0.67 DAN region = 'Midwest';
        ANDERS region = 'South';

        /* Verkoopkanaal */
        ALS rand('uniform') < 0.45 DAN channel = 'Store';
        ANDERS channel = 'Online';

        /* Prijs- en mediadrijvers */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Gebeurtenisindicatoren en een irrelevante weersafleider */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        ALS rand('uniform') < 0.40 DAN promo = 'Yes';
        ANDERS promo = 'No';

        /* Pure ruisvoorspellers (geen echt effect) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Wekelijks verkochte eenheden uit een bekende structurele vergelijking */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        ALS units < 0 DAN units = 0;
        UITVOER;
    EINDE;
    BEWAREN region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
UITVOEREN;


NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


## 2. Profileer de gegevens

Voordat we gaan modelleren, bevestigen we dat de respons en de belangrijkste continue kandidaten op zinnige schalen liggen.

In [2]:
PROCEDURE GEMIDDELDEN GEGEVENS=demand n mean std MIN MAX maxdec=1;
    VARIABELE units price competitor adspend email;
    label units="Verkochte eenheden" price="Prijs" competitor="Concurrentieprijs"
          adspend="Advertentie-uitgaven" email="E-mailvolume";
    TITEL "Wekelijkse vraag en kandidaat-drijvers";
UITVOEREN;

                                         Wekelijkse vraag en kandidaat-drijvers                                         

                                                  The MEANS Procedure

 Variable    Label                        N        Mean     Std Dev     Minimum     Maximum
 ------------------------------------------------------------------------------------------
 units       Verkochte eenheden         100       874.8       136.3       508.6      1162.3
 price       Prijs                      100        14.0         3.4         8.0        19.9
 competitor  Concurrentieprijs          100        13.8         3.4         8.1        19.9
 adspend     Advertentie-uitgaven       100       990.6       726.9        50.0      3358.0
 email       E-mailvolume               100        45.5        26.4         0.0        99.0
 ------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. Stapsgewijze selectie beoordeeld op SBC

Stapsgewijze selectie voegt effecten afwisselend toe en verwijdert ze, hier beoordeeld op het **Schwarz Bayesian Criterion (SBC)** voor zowel de opnametoets (`select=sbc`) als de uiteindelijke modelkeuze (`choose=sbc`). SBC bestraft complexiteit zwaarder dan AIC en geeft de voorkeur aan slankere modellen.

Belangrijke statements en opties:

- `CLASS region channel promo / param=reference` declareert de categorische voorspellers met referentiecel-codering.
- `selection=stepwise(select=sbc choose=sbc)` stuurt de zoektocht aan.
- `details=summary` drukt de stapsgewijze selectiesamenvatting af; `stb` voegt gestandaardiseerde coëfficiënten toe zodat effecten op verschillende schalen vergelijkbaar zijn.
- `output out=demand_scored p=predicted r=residual` bewaart gefitte waarden en residuen voor verdere scoring.

Lees de **Stepwise Selection Summary** als het spoor van de zoektocht: het start bij het volledige model met 12 effecten en *verwijdert* effecten één voor één, waarbij `noise1`, `noise2`, `temp_f`, het `Region=Midwest`-contrast en `noise3` achtereenvolgens vallen, omdat elke verwijdering de SBC verlaagt. Wat overblijft in de tabel **Parameter Estimates** is het gekozen model.

In [3]:
PROCEDURE glmselect GEGEVENS=demand seed=20260531;
    KLASSE region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(SELECTEREN=sbc choose=sbc)
          details=summary stb;
    UITVOER out=demand_scored p=predicted r=residual;
    label units="Verkochte eenheden" region="Regio" channel="Kanaal" promo="Actie"
          price="Prijs" competitor="Concurrentieprijs" adspend="Advertentie-uitgaven"
          email="E-mailvolume" temp_f="Temperatuur (F)" holiday="Feestdag"
          noise1="Ruis 1" noise2="Ruis 2" noise3="Ruis 3";
    TITEL "Stapsgewijze selectie van vraagdrijvers (SBC)";
UITVOEREN;

                                         Wekelijkse vraag en kandidaat-drijvers                                         

The GLMSELECT Procedure


Dependent Variable: UNITS Verkochte eenheden


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                 Stepwise Selection Summary                                                 

    Step    Action           Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  ---------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed           Ruis 1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.8766   12.0136
    


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. LASSO met kruisvalidatie

De LASSO krimpt coëfficiënten richting nul en voert tegelijk selectie en regularisatie uit. In plaats van te stoppen bij een vast criterium laten we **5-voudige kruisvalidatie** het punt op het LASSO-pad kiezen met de beste fout buiten de steekproef.

- `selection=lasso(choose=cv stop=none)` traceert het volledige LASSO-pad en selecteert de CV-optimale stap.
- `cvmethod=random(5)` wijst waarnemingen toe aan 5 willekeurige folds.

De **LASSO Selection Summary** toont de volgorde waarin effecten binnenkomen naarmate de straf versoepelt: `price` eerst, dan `adspend`, `competitor`, de regio Northeast, `email`, `promo` en `holiday` — alle zeven echte signalen komen binnen vóór enige afleider. Kruisvalidatie kiest vervolgens de stap waar de PRESS buiten de steekproef het laagst is, en de resulterende tabel **Parameter Estimates** behoudt precies de echte drijvers (plus een klein `Region=Midwest`-term) terwijl `temp_f`, `noise1`, `noise2` en `noise3` worden uitgesloten, die pas helemaal aan het einde van het pad binnenkomen.

In [4]:
PROCEDURE glmselect GEGEVENS=demand seed=20260531;
    KLASSE region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv STOPPEN=none)
          cvmethod=RANDOM(5);
    label units="Verkochte eenheden" region="Regio" channel="Kanaal" promo="Actie"
          price="Prijs" competitor="Concurrentieprijs" adspend="Advertentie-uitgaven"
          email="E-mailvolume" temp_f="Temperatuur (F)" holiday="Feestdag"
          noise1="Ruis 1" noise2="Ruis 2" noise3="Ruis 3";
    TITEL "LASSO met 5-voudige kruisvalidatie";
UITVOEREN;

                                         Wekelijkse vraag en kandidaat-drijvers                                         

The GLMSELECT Procedure


Dependent Variable: UNITS Verkochte eenheden


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                           LASSO Selection Summary                                                            

    Step    Action                Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  --------------------  ----


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. Voorwaartse selectie gevalideerd op een holdout

Een derde, complementaire strategie: bouw het model op met **voorwaartse selectie** (effecten komen alleen binnen, verlaten nooit), maar stop waar de prestatie op een onafhankelijke holdoutsteekproef het beste is — dit beschermt direct tegen overfitting.

- `partition fraction(validate=0.30)` reserveert willekeurig 30% van de rijen voor validatie, waardoor 70 trainings- en 30 validatiewaarnemingen overblijven.
- `selection=forward(select=aic choose=validate)` voegt effecten toe op basis van AIC maar kiest het uiteindelijke model op basis van de fout op de validatiesteekproef.

De tabel **Partition Fractions** bevestigt de 70/30-splitsing. Voorwaartse selectie voegt vervolgens effecten toe totdat de validatiefout niet meer verbetert; de acht effecten in de uiteindelijke tabel **Parameter Estimates** zijn precies de echte drijvers, waarbij de vier afleiders nooit worden toegelaten. Wanneer drie methoden gebaseerd op verschillende principes bij dezelfde drijvers uitkomen, is het model veel waarschijnlijker echt dan een artefact van één enkele selectieregel.

In [5]:
PROCEDURE glmselect GEGEVENS=demand seed=20260531;
    KLASSE region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(SELECTEREN=aic choose=validate);
    partition FRACTION(validate=0.30);
    label units="Verkochte eenheden" region="Regio" channel="Kanaal" promo="Actie"
          price="Prijs" competitor="Concurrentieprijs" adspend="Advertentie-uitgaven"
          email="E-mailvolume" temp_f="Temperatuur (F)" holiday="Feestdag"
          noise1="Ruis 1" noise2="Ruis 2" noise3="Ruis 3";
    TITEL "Voorwaartse selectie gevalideerd op een 30% holdout";
UITVOEREN;

                                         Wekelijkse vraag en kandidaat-drijvers                                         

The GLMSELECT Procedure


Dependent Variable: UNITS Verkochte eenheden


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                          Forward Selection Summary                                                           

    Step    Action                Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ---------


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. Resultaten interpreteren

Alle drie de selectiestrategieën herstellen **dezelfde set van echte vraagdrijvers** en verwijderen elke afleider. Rechtstreeks gelezen uit de tabellen **Parameter Estimates**:

- **Prijs** is de dominante drijver. De gestandaardiseerde coëfficiënt in het stapsgewijze model is **-0,70**, verreweg de grootste in magnitude, en de ruwe helling ligt tussen **-27,8** (stapsgewijs en LASSO) en **-28,6** (voorwaarts) eenheden per dollar — vrijwel precies de -28 waarmee de gegevens zijn gegenereerd. **Concurrentieprijs** beweegt de vraag de andere kant op met ongeveer **+14,4** in alle drie de fits, het substitutiegedrag dat een categoriemanager verwacht.
- **Advertentie-uitgaven** (ongeveer +0,062 eenheden per dollar) en **e-mailvolume** (ongeveer +1,2 eenheden per verzending) verhogen beide de verkoop, wat de mediarespons kwantificeert.
- **Promoties** en **feestdagen** dragen de grootste gebeurteniseffecten: het contrast `Promo=No` loopt ongeveer **-51 tot -57** ten opzichte van een gepromote week, en de feestdagstijging is ongeveer **+66 tot +76** eenheden. De **regio Northeast** (ongeveer +49 tot +55) en het **kanaal Online** (ongeveer +24 tot +32) dragen positieve basiseffecten.
- Cruciaal: elke afleider — `temp_f`, `noise1`, `noise2`, `noise3` — wordt **verwijderd** door stapsgewijze en voorwaartse selectie en uitgesloten uit het door CV gekozen LASSO-model. Elke methode herstelde het structurele signaal en negeerde de ruis.

De drie methoden zijn niet byte-voor-byte identiek: stapsgewijs (SBC) en de op holdout gevalideerde voorwaartse zoektocht komen uit op dezelfde acht effecten, terwijl de kruisgevalideerde LASSO bovendien een klein `Region=Midwest`-contrast behoudt (-8,3, standaardfout 8,3) — een verschil op de ruisvloer eerder dan een inhoudelijk meningsverschil. Dat een drijverlijst stapsgewijze SBC, kruisgevalideerde LASSO en op holdout gevalideerde voorwaartse selectie overleeft, is de echte conclusie: een bevinding die robuust is voor drie verschillende selectiefilosofieën is veel geloofwaardiger dan een die op één enkel criterium is afgestemd. Met `OUTPUT OUT=demand_scored` dat gefitte waarden en residuen vastlegt, breidt dezelfde workflow zich natuurlijk uit naar het scoren van de geplande prijs- en promotiekalender van volgend kwartaal.